In [1]:
import json
from pathlib import Path

import pandas as pd

In [7]:
def load_session_to_df(session_data, df):
    """
    Recibe un JSON ya cargado (dict) y agrega sus interacciones
    al DataFrame recibido.
    """

    try:
        session_id = session_data.get("session_id")
        user_id = session_data.get("user_id")
        created_at = session_data.get("created_at")
        session_mode = session_data.get("mode")
        session_use_rag = session_data.get("use_rag")

        # ---------------------------------
        # Manejo seguro de feedback null
        # ---------------------------------
        feedback = session_data.get("feedback") or {}

        csat = feedback.get("csat") or {}
        nps = feedback.get("nps") or {}
        resolution = feedback.get("resolution") or {}

        csat_score = csat.get("score")
        nps_score = nps.get("score")
        nps_category = nps.get("category")
        resolution_label = resolution.get("label")
        resolution_numeric = resolution.get("numeric")

        rows = []

        for idx, interaction in enumerate(session_data.get("interactions", []), start=1):

            titles = interaction.get("titles", [])
            scores = interaction.get("scores", [])

            row = {
                "session_id": session_id,
                "user_id": user_id,
                "created_at": created_at,
                "session_mode": session_mode,
                "session_use_rag": session_use_rag,

                "interaction_number": idx,
                "timestamp": interaction.get("timestamp"),
                "query": interaction.get("query"),
                "routed_query": interaction.get("routed_query"),
                "mode": interaction.get("mode"),
                "use_rag": interaction.get("use_rag"),
                "used_cache": interaction.get("used_cache"),
                "latency_ms": interaction.get("latency"),
                "results_count": interaction.get("results_count"),

                # Solo respuesta principal
                "title_response": titles[0] if len(titles) > 0 else None,
                "score_response": scores[0] if len(scores) > 0 else None,

                # Feedback
                "csat_score": csat_score,
                "nps_score": nps_score,
                "nps_category": nps_category,
                "resolution_label": resolution_label,
                "resolution_numeric": resolution_numeric
            }

            rows.append(row)

        temp_df = pd.DataFrame(rows)
        df = pd.concat([df, temp_df], ignore_index=True)

        return df

    except Exception as e:
        print(f"Error procesando sesión: {e}")
        return df

In [8]:
def load_all_sessions(folder_path):
    """
    Lee todos los archivos JSON cuyo nombre inicie con session_
    y consolida la información en un DataFrame.
    """

    folder_path = Path(folder_path)
    df = pd.DataFrame()

    files = list(folder_path.glob("session_*.json"))

    total_files = len(files)
    loaded_files = 0
    failed_files = 0

    print(f"Archivos encontrados: {total_files}\n")

    for i, file in enumerate(files, start=1):
        try:
            with open(file, "r", encoding="utf-8") as f:
                session_data = json.load(f)

            df = load_session_to_df(session_data, df)

            loaded_files += 1
            print(f"[{i}/{total_files}] OK -> {file.name}")

        except Exception as e:
            failed_files += 1
            print(f"[{i}/{total_files}] ERROR -> {file.name}: {e}")

    total_interactions = len(df)

    print("\n========== RESUMEN ==========")
    print(f"Archivos encontrados : {total_files}")
    print(f"Archivos leídos      : {loaded_files}")
    print(f"Archivos fallidos    : {failed_files}")
    print(f"Interacciones totales: {total_interactions}")
    print("=============================")

    return df

In [9]:
# Ruta carpeta origen
folder_path = Path(
    r"C:\Users\juans\OneDrive\Documentos\Maestria en Ingenieria y Analitica de Datos\Proyecto de Grado\CineMate AI\interactions"
)

# Cargar todos los archivos session_*.json
df_sessions = load_all_sessions(folder_path)

# Ver resultados
df_sessions.head()

Archivos encontrados: 351

[1/351] OK -> session_018bd500-cfad-4f2e-be92-f00b6a97f914.json
[2/351] OK -> session_022cd6bf-4c6e-40c2-a956-ff0002089c2f.json
[3/351] OK -> session_023b5b5d-ffb5-479e-b5c2-946af1a8407f.json
[4/351] OK -> session_02d32c7b-9cba-4722-829b-5b1658643d2b.json
[5/351] OK -> session_0324f93f-0a63-4883-992a-83f4bdce9ed1.json
[6/351] OK -> session_0532bfcd-2ca5-45f4-8a00-cefc232ce09d.json
[7/351] OK -> session_05f18ee5-0f82-46e6-858d-70869118f388.json
[8/351] OK -> session_064995ae-bfea-4c8f-b31d-2b7fa5749d2d.json
[9/351] OK -> session_06967f73-bcca-44c7-994d-af36009b7e2e.json
[10/351] OK -> session_073fc00a-e957-4973-aa12-1b018e2ed4fd.json
[11/351] OK -> session_07e5c2c6-ebdb-4305-933e-4d5e6af4d33b.json
[12/351] OK -> session_07f8e19b-0596-46fc-ab56-24aaf2410d24.json
[13/351] OK -> session_0819aab1-edc7-4e93-acc4-fdebde5fd1ad.json
[14/351] OK -> session_08b2d098-8bdd-41f3-88c4-e8f3a0dc1d4e.json
[15/351] OK -> session_0907057a-293c-4367-a22e-5fafeb376a72.json
[16/351

,session_id,user_id,created_at,session_mode,session_use_rag,interaction_number,timestamp,query,routed_query,mode,...,used_cache,latency_ms,results_count,title_response,score_response,csat_score,nps_score,nps_category,resolution_label,resolution_numeric
0,018bd500-cfad-4f2e-be92-f00b6a97f914,573171787521,2026-05-02T07:42:50.610244,DIRECT,False,1,2026-05-02T07:48:48.284812,películas de aventura épica,películas de aventura épica,DIRECT,...,False,229.763,5,Egypt: Secrets of the Pharaohs,0.2032,None,None,None,None,None
1,022cd6bf-4c6e-40c2-a956-ff0002089c2f,573156257476,2026-05-02T14:12:01.439573,DIRECT,False,1,2026-05-02T14:19:39.061673,"me gustó Gravity, algo parecido","me gustó Gravity, algo parecido",DIRECT,...,False,321.782,5,Bharathan Effect,0.2588,None,None,None,None,None
2,023b5b5d-ffb5-479e-b5c2-946af1a8407f,573174588659,2026-05-02T20:48:18.501169,DIRECT,False,1,2026-05-02T20:56:29.336779,acción con superhéroes pero más seria,acción con superhéroes pero más seria,DIRECT,...,False,382.660,5,Anche se è amore non si vede,0.2915,None,None,None,None,None
3,02d32c7b-9cba-4722-829b-5b1658643d2b,573131238842,2026-05-02T18:44:13.040285,DIRECT,False,1,2026-05-02T19:21:38.627476,terror psicológico con pocos personajes y ambi...,terror psicológico con pocos personajes y ambi...,DIRECT,...,False,2049.634,5,Utusan Iblis: Dia Yang Berada di Antara Kita,0.3324,None,None,None,None,None
4,0532bfcd-2ca5-45f4-8a00-cefc232ce09d,573167556912,2026-04-26T23:03:33.234190,DIRECT,False,1,2026-04-26T23:25:56.168851,recomiendame algo parecido a ready player one,recomiendame algo parecido a ready player one,DIRECT,...,False,1032.359,5,Deal or No Deal,0.3546,5,10,promoter,yes,3


In [10]:
# created_at -> solo fecha
df_sessions["created_at"] = pd.to_datetime(
    df_sessions["created_at"], errors="coerce"
).dt.strftime("%Y-%m-%d")

# timestamp -> solo hora:minuto
df_sessions["timestamp"] = pd.to_datetime(
    df_sessions["timestamp"], errors="coerce"
).dt.strftime("%H:%M")

# latency_ms -> minutos
df_sessions["latency_ms"] = df_sessions["latency_ms"] / 60000


df_sessions["user_id"] = (
    df_sessions["user_id"]
    .astype(str)
    .str.replace(r"^57(\d{3})(\d+)$", r"57 \1 \2", regex=True)
)

df_sessions.head()

,session_id,user_id,created_at,session_mode,session_use_rag,interaction_number,timestamp,query,routed_query,mode,...,used_cache,latency_ms,results_count,title_response,score_response,csat_score,nps_score,nps_category,resolution_label,resolution_numeric
0,018bd500-cfad-4f2e-be92-f00b6a97f914,57 317 1787521,2026-05-02,DIRECT,False,1,07:48,películas de aventura épica,películas de aventura épica,DIRECT,...,False,0.003829,5,Egypt: Secrets of the Pharaohs,0.2032,None,None,None,None,None
1,022cd6bf-4c6e-40c2-a956-ff0002089c2f,57 315 6257476,2026-05-02,DIRECT,False,1,14:19,"me gustó Gravity, algo parecido","me gustó Gravity, algo parecido",DIRECT,...,False,0.005363,5,Bharathan Effect,0.2588,None,None,None,None,None
2,023b5b5d-ffb5-479e-b5c2-946af1a8407f,57 317 4588659,2026-05-02,DIRECT,False,1,20:56,acción con superhéroes pero más seria,acción con superhéroes pero más seria,DIRECT,...,False,0.006378,5,Anche se è amore non si vede,0.2915,None,None,None,None,None
3,02d32c7b-9cba-4722-829b-5b1658643d2b,57 313 1238842,2026-05-02,DIRECT,False,1,19:21,terror psicológico con pocos personajes y ambi...,terror psicológico con pocos personajes y ambi...,DIRECT,...,False,0.034161,5,Utusan Iblis: Dia Yang Berada di Antara Kita,0.3324,None,None,None,None,None
4,0532bfcd-2ca5-45f4-8a00-cefc232ce09d,57 316 7556912,2026-04-26,DIRECT,False,1,23:25,recomiendame algo parecido a ready player one,recomiendame algo parecido a ready player one,DIRECT,...,False,0.017206,5,Deal or No Deal,0.3546,5,10,promoter,yes,3


In [11]:
# Ruta destino
output_file = Path(
    r"C:\Users\juans\OneDrive\Documentos\Maestria en Ingenieria y Analitica de Datos\Proyecto de Grado\CineMate AI\data\raw\chatbot_sessions_raw.csv"
)

# Guardar CSV
df_sessions.to_csv(output_file, index=False, encoding="utf-8-sig")

print("Archivo guardado en:")
print(output_file)

Archivo guardado en:
C:\Users\juans\OneDrive\Documentos\Maestria en Ingenieria y Analitica de Datos\Proyecto de Grado\CineMate AI\data\raw\chatbot_sessions_raw.csv
